In [1]:
# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_parquet("../data/final.parquet")

In [3]:
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [5]:
df_final = df.copy()

df_ordered = df_final.sort_values(["user_id", "order_id"])
orders_from_end = df_ordered.groupby("user_id").cumcount(ascending=False)

df_train = df_ordered.loc[orders_from_end >= 2].copy()

In [6]:
df_train

,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
4089398,431534,196,1,1,1,prior,5,4,15,28.0,Soda,77,7,soft drinks,beverages
4089399,431534,12427,2,1,1,prior,5,4,15,28.0,Original Beef Jerky,23,19,popcorn jerky,snacks
4089400,431534,10258,3,1,1,prior,5,4,15,28.0,Pistachios,117,19,nuts seeds dried fruit,snacks
4089401,431534,25133,4,1,1,prior,5,4,15,28.0,Organic String Cheese,21,16,packaged cheese,dairy eggs
4089402,431534,10326,5,0,1,prior,5,4,15,28.0,Organic Fuji Apples,24,4,fresh fruits,produce
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29908317,3154581,20073,9,0,206209,prior,1,3,11,NaN,Jazz Apple,24,4,fresh fruits,produce
29908318,3154581,13176,10,0,206209,prior,1,3,11,NaN,Bag of Organic Bananas,24,4,fresh fruits,produce
29908319,3154581,20842,11,0,206209,prior,1,3,11,NaN,Total 0% Greek Yogurt,120,16,yogurt,dairy eggs
29908320,3154581,23594,12,0,206209,prior,1,3,11,NaN,Medium Salsa,51,13,preserved dips spreads,pantry


In [7]:
# create baseline model
# get median baseline prediction
median_baseline = df_train.groupby('user_id')['days_since_prior_order'].median().reset_index().rename(
    columns={'days_since_prior_order':'median_baseline_pred'}
)

median_baseline

,user_id,median_baseline_pred
0,1,20.0
1,2,13.0
2,3,11.0
3,4,20.0
4,5,11.0
...,...,...
206204,206205,10.0
206205,206206,3.0
206206,206207,16.0
206207,206208,7.0


In [10]:
import numpy as np
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from xgboost import XGBClassifier

keys = ["user_id", "order_id"]
order_features = [
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
]

# One row per order, sorted in each user's order sequence.
orders = (
    df.sort_values(["user_id", "order_number", "order_id"])
    [keys + order_features]
    .drop_duplicates(keys)
    .reset_index(drop=True)
)

# Label an order by whether the following order was within 7 days.
orders["next_days"] = (
    orders.groupby("user_id")["days_since_prior_order"].shift(-1)
)
orders["from_end"] = orders.groupby("user_id").cumcount(ascending=False)

# Last two orders: second-to-last for validation, last for test.
# Training labels must end before validation, so omit the third-to-last
# order as a training example: its label comes from the validation order.
train_mask = (orders["from_end"] >= 3) & orders["next_days"].notna()
val_mask = (orders["from_end"] == 1) & orders["next_days"].notna()
test_mask = orders["from_end"] == 0

# Make sparse basket features from aisle and department IDs.
basket = df.groupby(keys, sort=False).agg(
    aisles=("aisle_id", lambda s: [f"a_{x}" for x in s.dropna().unique()]),
    departments=(
        "department_id",
        lambda s: [f"d_{x}" for x in s.dropna().unique()],
    ),
)
basket["tokens"] = basket["aisles"] + basket["departments"]
basket = basket.reindex(pd.MultiIndex.from_frame(orders[keys]))
tokens = basket["tokens"].tolist()

# Fit category vocabulary on training orders only.
mlb = MultiLabelBinarizer(sparse_output=True)
basket_train = mlb.fit_transform(
    [tokens[i] for i in np.flatnonzero(train_mask)]
)
basket_val = mlb.transform(
    [tokens[i] for i in np.flatnonzero(val_mask)]
)
basket_test = mlb.transform(
    [tokens[i] for i in np.flatnonzero(test_mask)]
)

# Impute numeric fields and one-hot encode order day/hour.
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"),
     ["order_number", "days_since_prior_order"]),
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     ["order_dow", "order_hour_of_day"]),
])

X_train = hstack([
    preprocessor.fit_transform(orders.loc[train_mask, order_features]),
    basket_train,
], format="csr")
X_val = hstack([
    preprocessor.transform(orders.loc[val_mask, order_features]),
    basket_val,
], format="csr")
X_test = hstack([
    preprocessor.transform(orders.loc[test_mask, order_features]),
    basket_test,
], format="csr")

y_train = orders.loc[train_mask, "next_days"].le(7).astype("int8")
y_val = orders.loc[val_mask, "next_days"].le(7).astype("int8")

model_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)
model_xgb.fit(X_train, y_train)

val_predictions = model_xgb.predict(X_val)
print("Validation accuracy:", accuracy_score(y_val, val_predictions))

test_predictions = orders.loc[test_mask, keys].copy()
test_predictions["predicted_next_order_within_7_days"] = model_xgb.predict(X_test)

Validation accuracy: 0.7005950273751388
